# Sleep Apnea Multimodal — GPU training on Colab

Runs the 30 s window condition (Experiment 2) for the remaining seed on a T4 GPU.

**Before you start**, upload `research_cache.zip` to your Google Drive under
`MyDrive/research/`. That zip must contain, at its root:

```
scripts/                        all .py files
config/                         model.json, policy.json
artifacts/signals/*.npy         the 78 per-record caches (~414 MB)
artifacts/windows/*.csv         windows_10s.csv, windows_30s.csv
artifacts/normalization/norm_stats.json
```

Everything else (train/test outputs) is regenerated here and written back to Drive
so a disconnect does not lose finished seeds.

## 0. Check the GPU

If this does not print a Tesla T4 (or similar), go to **Runtime -> Change runtime type -> T4 GPU**
and rerun. Training on CPU here would be slower than the laptop.

In [ ]:
import torch, platform
print('torch', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))
    print('memory GB:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
else:
    raise SystemExit('No GPU. Set Runtime -> Change runtime type -> T4 GPU.')

## 1. Mount Drive and copy the cache

Copies from Drive to `/content` because Drive reads are slow and would starve the
data loader.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, zipfile, shutil, pathlib

ZIP = '/content/drive/MyDrive/research/research_cache.zip'
ROOT = pathlib.Path('/content/Research_v2')

assert os.path.exists(ZIP), f'not found: {ZIP}'
if ROOT.exists():
    shutil.rmtree(ROOT)
ROOT.mkdir(parents=True)
with zipfile.ZipFile(ZIP) as z:
    z.extractall(ROOT)
print('extracted to', ROOT)
os.system(f'cd {ROOT} && ls && du -sh artifacts/signals')

## 2. Verify the cache before spending GPU time

Checks shapes, label parity, and normalization provenance. If this fails, the zip is
incomplete — do not start training.

In [ ]:
os.chdir(ROOT)
!python scripts/08_dataset.py 2024 30

## 3. Train seed 2024 at 30 s

`--workers 2` feeds the GPU from separate processes; on CPU this would be pointless.

**If the session disconnects**, rerun cells 1-3 and start this cell again with the
same command. `resume_<model>.pt` lets it continue from the last finished epoch
instead of restarting the model.

Order puts the cheap models first so you see results quickly.

In [ ]:
!python scripts/15_train.py --models resp_only,ecg_only,proposed,simple_fusion \
    --seed 2024 --duration 30 --device cuda --workers 2

## 4. Test with the frozen models

Reads `frozen.json` written by the cell above and applies the validation thresholds
unchanged. Test is read once per seed, per the Stage 6 protocol.

In [ ]:
!python scripts/16_test.py --seed 2024 --duration 30 --device cuda --workers 2

## 5. Copy results back to Drive

Run this even after a crash: finished models are already on disk.

In [ ]:
DEST = '/content/drive/MyDrive/research/results'
os.makedirs(DEST, exist_ok=True)
for sub in ('train/seed2024_30s', 'test/seed2024_30s'):
    src = ROOT / 'artifacts' / sub
    if src.exists():
        shutil.copytree(src, pathlib.Path(DEST) / sub, dirs_exist_ok=True)
        print('copied', sub)
print('\nfiles now on Drive:')
os.system(f'find {DEST} -maxdepth 2 -type f | head -40')

## 6. After it finishes

1. Download the two folders from Drive into the local repo at the same paths.
2. Note in `LOG_KERJA.md` that seed 2024 ran on a T4 GPU: floating-point results can
   differ marginally from CPU runs, so the device must be recorded.
3. Regenerate the aggregate tables locally:

```powershell
python scripts\19_aggregate.py --seeds 42,1337,2024 --duration 30
python scripts\20_aggregate_test.py --seeds 42,1337,2024 --duration 30
python scripts\10_report.py
```